# Accessing Data
- import.py
- relative ranking csv
- holiday function

## Import.py

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

## Holiday Function

In [ ]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

## Relative ranking mean, StD, variance

In [ ]:
import pandas as pd

rank = pd.read_csv(
    "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/full_nsw_relative_rank.csv"
)


# Plotting only the public holiday relative rank
- mean, variance and StD

## Mean
- mean relative rank  = y axis
- year = x axis
- each line represents a time block

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_mean_relative_rank_per_station(
    csv_path,
    out_base="/home/565/pv3484/aus_substation_electricity/figures/mean_relative_rank"
):
    """
    Reads the long-form daily relative-rank CSV and produces
    one plot per station per holiday, showing mean relative rank
    by year for the holiday ONLY (not the ±30-day window).
    """

    df = pd.read_csv(csv_path, parse_dates=["date"])

    # Filter to holiday-only rows
    df_h = df[df["is_holiday"] == True].copy()

    # Time blocks to plot
    block_cols = {
        "00–04": "00_04_mean",
        "04–10": "04_10_mean",
        "10–15": "10_15_mean",
        "15–20": "15_20_mean",
        "20–24": "20_24_mean",
    }

    # Colours for consistency
    colours = {
        "00–04": "red",
        "04–10": "orange",
        "10–15": "teal",
        "15–20": "blue",
        "20–24": "purple",
    }

    # Loop through holidays
    for holiday in df_h["holiday"].unique():

        holiday_folder = os.path.join(out_base, holiday.replace(" ", "_"))
        os.makedirs(holiday_folder, exist_ok=True)

        df_hol = df_h[df_h["holiday"] == holiday]

        # Loop through stations
        for station in df_hol["station_code"].unique():

            df_s = df_hol[df_hol["station_code"] == station].sort_values("year")

            # --- NEW: full station name for title ---
            full_name = df_s["station_name"].iloc[0]

            plt.figure(figsize=(10, 6))

            # Plot each block
            for label, col in block_cols.items():
                plt.plot(
                    df_s["year"],
                    df_s[col],
                    marker="o",
                    linewidth=2,
                    color=colours[label],
                    label=label
                )

            # --- NEW: title with full station name ---
            plt.title(f"{holiday} — Mean Relative Rank\n{full_name}", fontsize=16)

            plt.xlabel("Year", fontsize=14)
            plt.ylabel("Mean Relative Rank", fontsize=14)
            plt.ylim(0, 1)
            plt.grid(alpha=0.3)

            # --- NEW: legend below the plot ---
            plt.legend(
                title="Time Block (24hr)",
                fontsize=12,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.15),
                ncol=5,
                frameon=False
            )

            plt.tight_layout()

            out_path = os.path.join(holiday_folder, f"{station}.png")
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close()

    print("All plots saved.")


In [ ]:
plot_mean_relative_rank_per_station(
    csv_path="/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/full_nsw_relative_rank.csv"
)


2015 new year's day is empty from 1/01/15 - 4/01/15 due to NaN's in the raw data

## Variance

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_variance_relative_rank_per_station(
    csv_path,
    out_base="/home/565/pv3484/aus_substation_electricity/figures/variance_relative_rank"
):
    """
    Reads the long-form daily relative-rank CSV and produces
    one plot per station per holiday, showing variance of relative rank
    by year for the holiday ONLY (not the ±30-day window).
    """

    df = pd.read_csv(csv_path, parse_dates=["date"])

    # Filter to holiday-only rows
    df_h = df[df["is_holiday"] == True].copy()

    # Time blocks to plot (variance columns)
    block_cols = {
        "00–04": "00_04_var",
        "04–10": "04_10_var",
        "10–15": "10_15_var",
        "15–20": "15_20_var",
        "20–24": "20_24_var",
    }

    # Colours for consistency
    colours = {
        "00–04": "red",
        "04–10": "orange",
        "10–15": "teal",
        "15–20": "blue",
        "20–24": "purple",
    }

    # Loop through holidays
    for holiday in df_h["holiday"].unique():

        holiday_folder = os.path.join(out_base, holiday.replace(" ", "_"))
        os.makedirs(holiday_folder, exist_ok=True)

        df_hol = df_h[df_h["holiday"] == holiday]

        # Loop through stations
        for station in df_hol["station_code"].unique():

            df_s = df_hol[df_hol["station_code"] == station].sort_values("year")

            # Full station name for title
            full_name = df_s["station_name"].iloc[0]

            plt.figure(figsize=(10, 6))

            # Plot each block
            for label, col in block_cols.items():
                plt.plot(
                    df_s["year"],
                    df_s[col],
                    marker="o",
                    linewidth=2,
                    color=colours[label],
                    label=label
                )

            # Title with full station name
            plt.title(f"{holiday} — Variance of Relative Rank\n{full_name}", fontsize=16)

            plt.xlabel("Year", fontsize=14)
            plt.ylabel("Variance of Relative Rank", fontsize=14)
            plt.grid(alpha=0.3)

            # Legend below the plot
            plt.legend(
                title="Time Block (24hr)",
                fontsize=12,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.15),
                ncol=5,
                frameon=False
            )

            plt.tight_layout()

            out_path = os.path.join(holiday_folder, f"{station}.png")
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close()

    print("All variance plots saved.")


In [ ]:
plot_variance_relative_rank_per_station(
    csv_path="/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/full_nsw_relative_rank.csv"
)


## StD

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_std_relative_rank_per_station(
    csv_path,
    out_base="/home/565/pv3484/aus_substation_electricity/figures/StD_relative_rank"
):
    """
    Reads the long-form daily relative-rank CSV and produces
    one plot per station per holiday, showing standard deviation of relative rank
    by year for the holiday ONLY (not the ±30-day window).
    """

    df = pd.read_csv(csv_path, parse_dates=["date"])

    # Filter to holiday-only rows
    df_h = df[df["is_holiday"] == True].copy()

    # Time blocks to plot (StD columns)
    block_cols = {
        "00–04": "00_04_std",
        "04–10": "04_10_std",
        "10–15": "10_15_std",
        "15–20": "15_20_std",
        "20–24": "20_24_std",
    }

    # Colours for consistency
    colours = {
        "00–04": "red",
        "04–10": "orange",
        "10–15": "teal",
        "15–20": "blue",
        "20–24": "purple",
    }

    # Loop through holidays
    for holiday in df_h["holiday"].unique():

        holiday_folder = os.path.join(out_base, holiday.replace(" ", "_"))
        os.makedirs(holiday_folder, exist_ok=True)

        df_hol = df_h[df_h["holiday"] == holiday]

        # Loop through stations
        for station in df_hol["station_code"].unique():

            df_s = df_hol[df_hol["station_code"] == station].sort_values("year")

            # Full station name for title
            full_name = df_s["station_name"].iloc[0]

            plt.figure(figsize=(10, 6))

            # Plot each block
            for label, col in block_cols.items():
                plt.plot(
                    df_s["year"],
                    df_s[col],
                    marker="o",
                    linewidth=2,
                    color=colours[label],
                    label=label
                )

            # Title with full station name
            plt.title(f"{holiday} — Standard Deviation of Relative Rank\n{full_name}", fontsize=16)

            plt.xlabel("Year", fontsize=14)
            plt.ylabel("Standard Deviation of Relative Rank", fontsize=14)
            plt.grid(alpha=0.3)

            # Legend below the plot
            plt.legend(
                title="Time Block (24hr)",
                fontsize=12,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.15),
                ncol=5,
                frameon=False
            )

            plt.tight_layout()

            out_path = os.path.join(holiday_folder, f"{station}.png")
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close()

    print("All standard deviation plots saved.")


In [ ]:
plot_std_relative_rank_per_station(
    csv_path="/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/full_nsw_relative_rank.csv"
)


### 2x2 panel for powerpoint

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_std_panels_for_holiday(
    df,
    holiday,
    stations=["BLAKE", "GATES", "LIDCO", "CHATS"]
):
    """
    Produces a 2x2 panel of standard deviation plots for the given holiday
    for the four specified stations. Output is shown inline.
    """

    # Filter to holiday-only rows
    df_h = df[(df["is_holiday"] == True) & (df["holiday"] == holiday)].copy()

    # Standard deviation columns
    block_cols = {
        "00–04": "00_04_std",
        "04–10": "04_10_std",
        "10–15": "10_15_std",
        "15–20": "15_20_std",
        "20–24": "20_24_std",
    }

    colours = {
        "00–04": "red",
        "04–10": "orange",
        "10–15": "teal",
        "15–20": "blue",
        "20–24": "purple",
    }

    # 2x2 figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for ax, station in zip(axes, stations):

        df_s = df_h[df_h["station_code"] == station].sort_values("year")

        if df_s.empty:
            ax.set_title(f"{station} (no data)")
            ax.axis("off")
            continue

        full_name = df_s["station_name"].iloc[0]

        # Plot each time block
        for label, col in block_cols.items():
            ax.plot(
                df_s["year"],
                df_s[col],
                marker="o",
                linewidth=2,
                color=colours[label],
                label=label
            )

        ax.set_title(full_name, fontsize=14)
        ax.set_xlabel("Year")
        ax.set_ylabel("Standard Deviation of Relative Rank")
        ax.grid(alpha=0.3)

    # Shared legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        title="Time Block (24hr)",
        loc="upper center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=5,
        frameon=False
    )

    fig.suptitle(f"{holiday} — Standard Deviation of Relative Rank", fontsize=18, y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.close()


In [ ]:
# Load once
df = pd.read_csv(
    "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/full_nsw_relative_rank.csv",
    parse_dates=["date"]
)

# Get all holidays in the dataset
holidays = df[df["is_holiday"] == True]["holiday"].unique()

# Loop and plot
for h in holidays:
    plot_std_panels_for_holiday(df, h)


# Metadata tables

In [ ]:
stations = ["BLAKEHURST", "GATESHEAD", "CHATSWOOD", "LIDCOMBE"]

info.loc[info["Name"].str.upper().isin(stations), 
         ["Name", "Residential", "Industrial"]]


In [ ]:
rank.head()

import pandas as pd

stations_df = (
    rank[["station_code", "station_name"]]
    .drop_duplicates()
    .sort_values("station_code")
)

out_path = "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/all_substation_names.txt"
stations_df.to_csv(out_path, index=False, header=True, sep="\t")

# Plotting relative rank vs mean daily temperature
- scatter plots
- weekdays/weekends in grey, years in colours

## Weekdays only for the 30+/- AND public holiday

### Filtering for weekdays in csv file

In [ ]:
#converting date column into datetime
rank["date"] = pd.to_datetime(rank["date"], errors="coerce")

#filtering for weekdays
rank_weekdays = rank[rank["date"].dt.weekday < 5].copy()

In [ ]:
rank_weekdays["date"].dt.weekday.unique()


### Prep temperature plotting

In [ ]:
#preparing the temperature data (converting half hourly to daily)

# keep only numeric columns (drops t2m_bin)
obs_numeric = obs.select_dtypes(include="number")

# now resample safely
obs_daily = (
    obs_numeric
    .resample("D")
    .mean()
    .reset_index()
    .rename(columns={"t2m": "temp"})
)

#renaming 'index' to 'date'
obs_daily = obs_daily.rename(columns={"index": "date"})


In [ ]:
#merging obs_daily into rank

#produce a new column in 'rank' with the mean daily temperature
rank = rank.merge(
    obs_daily[["date", "temp"]],
    on="date",
    how="left"
)



#### Plotting function

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

def plot_temp_vs_meanrank_with_window(
    df,
    holiday,
    station,
    window=60,
    save_path=None
):

    """
    For a given holiday + station:
    - ONE coloured dot per year (holiday only)
    - Grey dots = weekdays within ±window days of the holiday (same year)
    - Legend = one dot per year with temperature in brackets
    - Colours = 14 distinct colours from tab20b
    """

    # Filter to this station
    df_s = df[df["station_code"] == station].copy()

    # If station has no data at all
    if df_s.empty:
        print(f"Skipping {holiday} for {station} — station has no data.")
        return

    # Extract full station name
    full_name = df_s["station_name"].iloc[0]

    # Filter to this holiday for this station
    df_h_raw = df_s[df_s["holiday"] == holiday]

    # If this holiday does not exist for this station
    if df_h_raw.empty:
        print(f"Skipping {holiday} for {station} — holiday not present.")
        return

    # ---- AGGREGATE HOLIDAY ROWS TO ONE ROW PER YEAR ----
    df_h = (
        df_h_raw
        .groupby("year", as_index=False)
        .agg({
            "temp": "mean",
            "00_04_mean": "mean",
            "04_10_mean": "mean",
            "10_15_mean": "mean",
            "15_20_mean": "mean",
            "20_24_mean": "mean",
            "date": "first"
        })
    )

    # 5‑panel figure
    fig, axes = plt.subplots(1, 5, figsize=(22, 6.5), sharey=True)

    # ---- BUILD 14 DISTINCT COLOURS FROM tab20b ----
    base_cmap = plt.cm.get_cmap("tab20")
    year_colors = [base_cmap(i) for i in range(len(df_h))]

    years = sorted(df_h["year"].unique())
    color_map = {y: year_colors[i] for i, y in enumerate(years)}

    # Time blocks
    block_cols = {
        "00–04": "00_04_mean",
        "04–10": "04_10_mean",
        "10–15": "10_15_mean",
        "15–20": "15_20_mean",
        "20–24": "20_24_mean",
    }

    # ---- PLOTTING ----
    for ax, (label, col) in zip(axes, block_cols.items()):

        # --- GREY WEEKDAY POINTS (per year) ---
        for _, row in df_h.iterrows():
            d0 = row["date"]
            y = row["year"]

            mask = (
                (df_s["year"] == y) &
                (df_s["date"] >= d0 - pd.Timedelta(days=window)) &
                (df_s["date"] <= d0 + pd.Timedelta(days=window)) &
                (df_s["date"].dt.weekday < 5) &
                (df_s["is_holiday"] == False)
            )

            df_win = df_s[mask]

            ax.scatter(
                df_win["temp"],
                df_win[col],
                color="lightgrey",
                alpha=0.35,
                s=18
            )

        # --- COLOURED HOLIDAY POINTS (ONE PER YEAR) ---
        ax.scatter(
            df_h["temp"],
            df_h[col],
            c=[color_map[y] for y in df_h["year"]],
            s=75,
            edgecolor="black"
        )

        ax.set_title(label)
        ax.set_xlabel("Mean Daily Temperature (°C)")

    axes[0].set_ylabel("Mean Relative Rank (0–1)")

    # ---- CUSTOM DOT LEGEND WITH TEMPERATURE ----
    legend_handles = []
    for _, row in df_h.iterrows():
        year = row["year"]
        temp = row["temp"]
        label = f"{year} ({temp:.1f}°C)"

        handle = Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor=color_map[year],
            markeredgecolor='black',
            markersize=11,
            label=label
        )
        legend_handles.append(handle)

    fig.legend(
        handles=legend_handles,
        title="Year (Mean Temp)",
        loc="lower center",
        ncol=7,
        frameon=False,
        bbox_to_anchor=(0.5, -0.08),
        fontsize=12,
        title_fontsize=13
    )

    plt.suptitle(
        f"{holiday} — {full_name} — Mean Rank vs Temperature",
        fontsize=16
    )

    plt.subplots_adjust(bottom=0.22)
        # Save figure if a path is provided
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.close


In [ ]:
import os

base_dir = "/home/565/pv3484/aus_substation_electricity/figures/mean_weekday_temp"

stations = rank["station_code"].unique()

# Loop through each station
for st in stations:

    df_st = rank[rank["station_code"] == st]

    # Only holidays that actually occur for this station
    holidays_st = df_st[df_st["is_holiday"] == True]["holiday"].unique()

    for h in holidays_st:

        print(f"Plotting {h} for {st}...")

        # Create folder for this holiday
        holiday_folder = os.path.join(base_dir, h.replace(" ", "_"))
        os.makedirs(holiday_folder, exist_ok=True)

        # Build output filename
        out_path = os.path.join(holiday_folder, f"{st}.png")

        # Call your plotting function, but with a save option
        plot_temp_vs_meanrank_with_window(
            rank,
            holiday=h,
            station=st,
            window=60,
            save_path=out_path
        )


## Weekends only for the 30+/- AND public holiday

### Filtering for weekends only

In [ ]:
#converting date column into datetime
rank["date"] = pd.to_datetime(rank["date"], errors="coerce")

#filtering for weekdays
rank_weekends = rank[rank["date"].dt.weekday >= 5].copy()

In [ ]:
rank_weekends["date"].dt.weekday.unique()


### Plotting

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

def plot_temp_vs_meanrank_with_window(
    df,
    holiday,
    station,
    window=60,
    save_path=None
):

    """
    For a given holiday + station:
    - ONE coloured dot per year (holiday only)
    - Grey dots = weekendsx within ±window days of the holiday (same year)
    - Legend = one dot per year with temperature in brackets
    - Colours = 14 distinct colours from tab20b
    """

    # Filter to this station
    df_s = df[df["station_code"] == station].copy()

    # If station has no data at all
    if df_s.empty:
        print(f"Skipping {holiday} for {station} — station has no data.")
        return

    # Extract full station name
    full_name = df_s["station_name"].iloc[0]

    # Filter to this holiday for this station
    df_h_raw = df_s[df_s["holiday"] == holiday]

    # If this holiday does not exist for this station
    if df_h_raw.empty:
        print(f"Skipping {holiday} for {station} — holiday not present.")
        return

    # ---- AGGREGATE HOLIDAY ROWS TO ONE ROW PER YEAR ----
    df_h = (
        df_h_raw
        .groupby("year", as_index=False)
        .agg({
            "temp": "mean",
            "00_04_mean": "mean",
            "04_10_mean": "mean",
            "10_15_mean": "mean",
            "15_20_mean": "mean",
            "20_24_mean": "mean",
            "date": "first"
        })
    )

    # 5‑panel figure
    fig, axes = plt.subplots(1, 5, figsize=(22, 6.5), sharey=True)

    # ---- BUILD 14 DISTINCT COLOURS FROM tab20b ----
    base_cmap = plt.cm.get_cmap("tab20")
    year_colors = [base_cmap(i) for i in range(len(df_h))]

    years = sorted(df_h["year"].unique())
    color_map = {y: year_colors[i] for i, y in enumerate(years)}

    # Time blocks
    block_cols = {
        "00–04": "00_04_mean",
        "04–10": "04_10_mean",
        "10–15": "10_15_mean",
        "15–20": "15_20_mean",
        "20–24": "20_24_mean",
    }

    # ---- PLOTTING ----
    for ax, (label, col) in zip(axes, block_cols.items()):

        # --- GREY WEEKENDS POINTS (per year) ---
        for _, row in df_h.iterrows():
            d0 = row["date"]
            y = row["year"]

            mask = (
                (df_s["year"] == y) &
                (df_s["date"] >= d0 - pd.Timedelta(days=window)) &
                (df_s["date"] <= d0 + pd.Timedelta(days=window)) &
                (df_s["date"].dt.weekday >= 5) &
                (df_s["is_holiday"] == False)
            )

            df_win = df_s[mask]

            ax.scatter(
                df_win["temp"],
                df_win[col],
                color="lightgrey",
                alpha=0.35,
                s=18
            )

        # --- COLOURED HOLIDAY POINTS (ONE PER YEAR) ---
        ax.scatter(
            df_h["temp"],
            df_h[col],
            c=[color_map[y] for y in df_h["year"]],
            s=75,
            edgecolor="black"
        )

        ax.set_title(label)
        ax.set_xlabel("Mean Daily Temperature (°C)")

    axes[0].set_ylabel("Mean Relative Rank (0–1)")

    # ---- CUSTOM DOT LEGEND WITH TEMPERATURE ----
    legend_handles = []
    for _, row in df_h.iterrows():
        year = row["year"]
        temp = row["temp"]
        label = f"{year} ({temp:.1f}°C)"

        handle = Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor=color_map[year],
            markeredgecolor='black',
            markersize=11,
            label=label
        )
        legend_handles.append(handle)

    fig.legend(
        handles=legend_handles,
        title="Year (Mean Temp)",
        loc="lower center",
        ncol=7,
        frameon=False,
        bbox_to_anchor=(0.5, -0.08),
        fontsize=12,
        title_fontsize=13
    )

    plt.suptitle(
        f"{holiday} — {full_name} — Mean Rank vs Temperature",
        fontsize=16
    )
    
    plt.subplots_adjust(bottom=0.22)
    
    # Save figure if a path is provided
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    
    plt.close()

In [ ]:
import os

base_dir = "/home/565/pv3484/aus_substation_electricity/figures/mean_weekend_temp"

stations = rank["station_code"].unique()

# Loop through each station
for st in stations:

    df_st = rank[rank["station_code"] == st]

    # Only holidays that actually occur for this station
    holidays_st = df_st[df_st["is_holiday"] == True]["holiday"].unique()

    for h in holidays_st:

        print(f"Plotting {h} for {st}...")

        # Create folder for this holiday
        holiday_folder = os.path.join(base_dir, h.replace(" ", "_"))
        os.makedirs(holiday_folder, exist_ok=True)

        # Build output filename
        out_path = os.path.join(holiday_folder, f"{st}.png")

        # Call your plotting function, but with a save option
        plot_temp_vs_meanrank_with_window(
            rank,
            holiday=h,
            station=st,
            window=60,
            save_path=out_path
        )


### powerpoint plotting

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

def plot_temp_vs_meanrank_with_window(
    df,
    holiday,
    station,
    window=60
):
    """
    Notebook‑only version:
    - Shows the 5‑panel figure inline
    - One coloured dot per year
    - Grey weekend dots ±window days
    - One legend per figure
    """

    # Filter to this station
    df_s = df[df["station_code"] == station].copy()
    if df_s.empty:
        print(f"Skipping {holiday} for {station} — station has no data.")
        return

    full_name = df_s["station_name"].iloc[0]

    # Filter to this holiday
    df_h_raw = df_s[df_s["holiday"] == holiday]
    if df_h_raw.empty:
        print(f"Skipping {holiday} for {station} — holiday not present.")
        return

    # Aggregate to one row per year
    df_h = (
        df_h_raw
        .groupby("year", as_index=False)
        .agg({
            "temp": "mean",
            "00_04_mean": "mean",
            "04_10_mean": "mean",
            "10_15_mean": "mean",
            "15_20_mean": "mean",
            "20_24_mean": "mean",
            "date": "first"
        })
    )

    # 5‑panel figure
    fig, axes = plt.subplots(1, 5, figsize=(22, 6.5), sharey=True)

    # Colours
    base_cmap = plt.cm.get_cmap("tab20")
    year_colors = [base_cmap(i) for i in range(len(df_h))]
    years = sorted(df_h["year"].unique())
    color_map = {y: year_colors[i] for i, y in enumerate(years)}

    # Time blocks
    block_cols = {
        "00–04": "00_04_mean",
        "04–10": "04_10_mean",
        "10–15": "10_15_mean",
        "15–20": "15_20_mean",
        "20–24": "20_24_mean",
    }

    # Plotting
    for ax, (label, col) in zip(axes, block_cols.items()):

        # Grey weekend points
        for _, row in df_h.iterrows():
            d0 = row["date"]
            y = row["year"]

            mask = (
                (df_s["year"] == y) &
                (df_s["date"] >= d0 - pd.Timedelta(days=window)) &
                (df_s["date"] <= d0 + pd.Timedelta(days=window)) &
                (df_s["date"].dt.weekday >= 5) &
                (~df_s["is_holiday"])
            )

            df_win = df_s[mask]

            ax.scatter(
                df_win["temp"],
                df_win[col],
                color="lightgrey",
                alpha=0.35,
                s=18
            )

        # Coloured holiday points
        ax.scatter(
            df_h["temp"],
            df_h[col],
            c=[color_map[y] for y in df_h["year"]],
            s=75,
            edgecolor="black"
        )

        ax.set_title(label)
        ax.set_xlabel("Mean Daily Temperature (°C)")

    axes[0].set_ylabel("Mean Relative Rank (0–1)")

    # Legend
    legend_handles = []
    for _, row in df_h.iterrows():
        year = row["year"]
        temp = row["temp"]
        label = f"{year} ({temp:.1f}°C)"

        handle = Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor=color_map[year],
            markeredgecolor='black',
            markersize=11,
            label=label
        )
        legend_handles.append(handle)

    fig.legend(
        handles=legend_handles,
        title="Year (Mean Temp)",
        loc="lower center",
        ncol=7,
        frameon=False,
        bbox_to_anchor=(0.5, -0.08),
        fontsize=12,
        title_fontsize=13
    )

    plt.suptitle(
        f"{holiday} — {full_name} — Mean Rank vs Temperature",
        fontsize=16
    )

    plt.subplots_adjust(bottom=0.22)
    plt.show()


In [ ]:
stations_to_plot = ["BLAKE", "LIDCO", "GATES", "CHATS"]

for st in stations_to_plot:
    df_st = rank[rank["station_code"] == st]
    holidays_st = df_st[df_st["is_holiday"]]["holiday"].unique()

    for h in holidays_st:
        print(f"Plotting {h} for {st}...")
        plot_temp_vs_meanrank_with_window(rank, holiday=h, station=st, window=60)


## Weekend vs Weekday vs Public Holiday vs temo
- weekend in light blue
- weekday in light orange
- public holiday in colour

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

def plot_temp_vs_meanrank_weekday_weekend(
    df,
    holiday,
    station,
    window=30,
    save_path=None
):
    """
    For a given holiday + station:
    - One coloured dot per year (holiday only)
    - Pastel orange = weekdays within ±30 days (same year)
    - Pastel blue   = weekends within ±30 days (same year)
    - Uses relative ranks already computed in the CSV (2-year rolling window)
    """

    # Filter to this station
    df_s = df[df["station_code"] == station].copy()
    if df_s.empty:
        print(f"Skipping {holiday} for {station} — station has no data.")
        return

    full_name = df_s["station_name"].iloc[0]

    # Holiday rows for this station
    df_h_raw = df_s[df_s["holiday"] == holiday]
    if df_h_raw.empty:
        print(f"Skipping {holiday} for {station} — holiday not present.")
        return

    # Aggregate holiday rows to one per year
    df_h = (
        df_h_raw
        .groupby("year", as_index=False)
        .agg({
            "temp": "mean",
            "00_04_mean": "mean",
            "04_10_mean": "mean",
            "10_15_mean": "mean",
            "15_20_mean": "mean",
            "20_24_mean": "mean",
            "date": "first"
        })
    )

    # 5‑panel figure
    fig, axes = plt.subplots(1, 5, figsize=(22, 6.5), sharey=True)

    # Pastel colours (equal visibility)
    pastel_weekday = "#fdbf6f"   # orange
    pastel_weekend = "#a6cee3"   # blue

    # Time blocks (AM/PM labels)
    block_cols = {
        "12 am – 4 am": "00_04_mean",
        "4 am – 10 am": "04_10_mean",
        "10 am – 3 pm": "10_15_mean",
        "3 pm – 8 pm": "15_20_mean",
        "8 pm – 12 am": "20_24_mean",
    }

    # Build colours for holiday dots
    base_cmap = plt.cm.get_cmap("tab20")
    year_colors = [base_cmap(i) for i in range(len(df_h))]
    years = sorted(df_h["year"].unique())
    color_map = {y: year_colors[i] for i, y in enumerate(years)}

    # ---- PLOTTING ----
    for ax, (label, col) in zip(axes, block_cols.items()):

        # Background weekday + weekend points
        for _, row in df_h.iterrows():
            d0 = row["date"]
            y = row["year"]

            # ±30-day window (CSV already contains 2-year rolling ranks)
            mask_base = (
                (df_s["year"] == y) &
                (df_s["date"] >= d0 - pd.Timedelta(days=window)) &
                (df_s["date"] <= d0 + pd.Timedelta(days=window)) &
                (~df_s["is_holiday"])
            )

            # Weekdays
            df_weekday = df_s[
                mask_base & (df_s["date"].dt.weekday < 5)
            ]

            # Weekends
            df_weekend = df_s[
                mask_base & (df_s["date"].dt.weekday >= 5)
            ]

            # Plot both with equal visibility
            ax.scatter(
                df_weekday["temp"], df_weekday[col],
                color=pastel_weekday, alpha=0.35, s=18
            )
            ax.scatter(
                df_weekend["temp"], df_weekend[col],
                color=pastel_weekend, alpha=0.35, s=18
            )

        # Holiday dots (one per year)
        ax.scatter(
            df_h["temp"],
            df_h[col],
            c=[color_map[y] for y in df_h["year"]],
            s=75,
            edgecolor="black"
        )

        ax.set_title(label)
        ax.set_xlabel("Mean Daily Temperature (°C)")

    axes[0].set_ylabel("Mean Relative Rank (0–1)")

    # ---- LEGEND ----
    legend_handles = [
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor=pastel_weekday, markeredgecolor='black',
               markersize=11, label="Weekday"),
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor=pastel_weekend, markeredgecolor='black',
               markersize=11, label="Weekend")
    ]

    for _, row in df_h.iterrows():
        year = row["year"]
        temp = row["temp"]
        label = f"{year} ({temp:.1f}°C)"
        legend_handles.append(
            Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=color_map[year],
                   markeredgecolor='black',
                   markersize=11, label=label)
        )

    fig.legend(
        handles=legend_handles,
        title="Legend",
        loc="lower center",
        ncol=7,
        frameon=False,
        bbox_to_anchor=(0.5, -0.08),
        fontsize=12,
        title_fontsize=13
    )

    plt.suptitle(
        f"{holiday} — {full_name} — Mean Rank vs Temperature",
        fontsize=16
    )

    plt.subplots_adjust(bottom=0.22)

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()


In [ ]:
stations_to_plot = ["BLAKE", "LIDCO", "CHATS", "GATES"]

all_holidays = rank[rank["is_holiday"]]["holiday"].unique()

for h in all_holidays:
    print(f"Plotting {h}...")
    plot_holiday_multistation(
        rank,
        holiday=h,
        stations=stations_to_plot,
        window=30
    )
